In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :memoryless

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [memoryless_model] Fitting chain 1 (tau=59)
[ Info: [memoryless] iter 1000/1000000 elapsed=5.5s, rate=0.026, mean=[1.043, 0.00016, 0.358], std=[0.0031, 0.000293, 0.0057] [ADAPT]
[ Info: [memoryless] iter 2000/1000000 elapsed=10.1s, rate=0.020, mean=[1.042, 0.00014, 0.380], std=[0.0056, 0.000218, 0.0247] [ADAPT]
[ Info: [memoryless] iter 3000/1000000 elapsed=13.9s, rate=0.022, mean=[1.029, 0.00014, 0.430], std=[0.0207, 0.000186, 0.0777] [ADAPT]
[ Info: [memoryless] iter 4000/1000000 elapsed=17.8s, rate=0.027, mean=[0.996, 0.00016, 0.491], std=[0.0571, 0.000170, 0.1170] [ADAPT]
[ Info: [memoryless] iter 5000/1000000 elapsed=21.8s, rate=0.042, mean=[0.937, 0.00026, 0.552], std=[0.1201, 0.000262, 0.1516] [ADAPT]
[ Info: [memoryless] iter 6000/1000000 elapsed=25.8s, rate=0.054, mean=[0.894, 0.00034, 0.598], std=[0.1392, 0.000290, 0.1663] [ADAPT]
[ Info: [memoryless] iter 7000/1000000 elapsed=29.7s, rate=0.062, mean=[0.867, 0.00038, 0.629], std=[0.1419, 0.000289, 0.1687] [ADAPT]
[ In